In [ ]:
import pandas as pd
import itertools

tea_df = pd.read_csv(r"H:\Projects\student-management-system\data\3.teachers_data.csv")
cls_df=  pd.read_csv(r"H:\Projects\student-management-system\data\4.classes.csv")
sub_df=  pd.read_csv(r"H:\Projects\student-management-system\data\6.subjects.csv")
gra_df=  pd.read_csv(r"H:\Projects\student-management-system\data\1.grades.csv")
print("teachers columns:", tea_df.columns.tolist())
print("classes columns:", cls_df.columns.tolist())
print("subjects columns:", sub_df.columns.tolist())
print("Grade columns:", gra_df.columns.tolist())

sub_df.head()

In [ ]:
DAYS = ["MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY", "SATURDAY"]

PERIODS_BY_DAY = {
    "MONDAY":    ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4", "PERIOD5", "PERIOD6", "PERIOD7", "PERIOD8"],
    "TUESDAY":   ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4", "PERIOD5", "PERIOD6", "PERIOD7", "PERIOD8"],
    "WEDNESDAY": ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4", "PERIOD5", "PERIOD6", "PERIOD7", "PERIOD8"],
    "THURSDAY":  ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4", "PERIOD5", "PERIOD6", "PERIOD7", "PERIOD8"],
    "FRIDAY":    ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4", "PERIOD5", "PERIOD6", "PERIOD7", "PERIOD8"],
    "SATURDAY":  ["PERIOD1", "PERIOD2", "PERIOD3", "PERIOD4"],
}

subjects_expanded = (
    sub_df
    .assign(gradeId=sub_df["gradeIds"].astype(str).str.split(","))
    .explode("gradeId")
)

subjects_expanded["gradeId"] = subjects_expanded["gradeId"].astype(int)


In [ ]:
import random

lessons = []
lesson_id = 1

for _, cls in cls_df.iterrows():
    class_id = cls["id"]
    grade_id = cls["gradeId"]

    # Teacher assigned to class
    teacher_row = tea_df[tea_df["classId"] == class_id]
    if teacher_row.empty:
        continue

    teacher_id = teacher_row.iloc[0]["id"]

    # Subjects for grade (cycled)
    subjects = subjects_expanded[
        subjects_expanded["gradeId"] == grade_id
    ]["name"].tolist()

    subject_index = 0

    for day in DAYS:
        periods = PERIODS_BY_DAY[day]

        # Random FREE period per day
        free_period = random.choice(periods)

        for period in periods:
            if period == free_period:
                lessons.append({
                    "id": lesson_id,
                    "gradeId": grade_id,
                    "classId": class_id,
                    "subject": "",
                    "teacherId": None,
                    "day": day,
                    "period": period
                })
            else:
                lessons.append({
                    "id": lesson_id,
                    "gradeId": grade_id,
                    "classId": class_id,
                    "subject": subjects[subject_index % len(subjects)],
                    "teacherId": teacher_id,
                    "day": day,
                    "period": period
                })
                subject_index += 1

            lesson_id += 1


In [ ]:
lessons_df = pd.DataFrame(lessons)
lessons_df.head(10)


In [ ]:
sub_df.head(1)

In [ ]:
sub_df.rename(columns={"name": "subject"}, inplace=True)

In [ ]:
sub_df.head(5)

In [ ]:
lessons_df = pd.merge(lessons_df,sub_df, on="subject", how="left")
lessons_df

In [ ]:
lessons_df.rename(columns={"id_x" : "id", "id_y" : "subjectId"}, inplace=True)

In [ ]:
lessons_df

In [ ]:
lessons_df = lessons_df.drop(columns=["subject", "gradeIds"])
lessons_df

In [ ]:
lessons_df.isnull().sum()

In [ ]:
lessons_df = lessons_df.dropna(subset=["teacherId"])

In [ ]:
lessons_df = lessons_df.reset_index(drop=True)

In [ ]:
lessons_df["id"] = range(1, len(lessons_df) + 1)
lessons_df

In [ ]:
lessons_df['subjectId'] = lessons_df['subjectId'].astype(int)

In [ ]:
lessons_df

In [ ]:
lessons_df.to_csv(
    r"H:\Projects\student-management-system\data\8.lessons.csv",
    index=False
)
